In [2]:
"""
Part 1 - Data Acquisition, Cleaning, and Exploratory Analysis
Applied AI & ML Essentials Capstone - Synthetic E-Commerce Dataset

Run top-to-bottom in Jupyter/Colab. Each "# TASK n" block corresponds
to one numbered task in the assignment. Plots are saved to ./plots/
and the cleaned dataset is saved to ./cleaned_data.csv
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
os.makedirs("plots", exist_ok=True)

# ------------------------------------------------------------------
# STEP 0: Generate the synthetic e-commerce dataset (this creates the
# raw CSV that Task 1 loads). Run this once to produce ecommerce_raw.csv
# ------------------------------------------------------------------
n = 800

order_id = np.arange(1, n + 1)
customer_age = np.clip(np.random.normal(35, 10, n), 18, 80).round(0)

# Positively skewed target (purchase amount) with a few extreme outliers
purchase_amount = np.random.lognormal(mean=3.5, sigma=0.6, size=n)
purchase_amount[np.random.choice(n, 8, replace=False)] *= np.random.uniform(4, 7, 8)  # outliers
purchase_amount = purchase_amount.round(2)

quantity = np.random.poisson(3, n) + 1
quantity[np.random.choice(n, 6, replace=False)] += np.random.randint(15, 25, 6)  # outliers

# discount_percent stored as messy strings (needs dtype correction)
discount_vals = np.random.choice([0, 5, 10, 15, 20, 25], n)
discount_percent = [f"{d}%" if d > 0 else "0%" for d in discount_vals]
noise_idx = np.random.choice(n, 12, replace=False)
for i in noise_idx:
    discount_percent[i] = "N/A"

# customer_rating: numeric, >20% missing on purpose
customer_rating = np.clip(np.random.normal(4.0, 0.8, n), 1, 5).round(1)
missing_idx = np.random.choice(n, int(n * 0.27), replace=False)
customer_rating = customer_rating.astype(object)
for i in missing_idx:
    customer_rating[i] = np.nan

# delivery_days: monotonic non-linear (sqrt) function of purchase_amount + noise
delivery_days = (2 + np.sqrt(purchase_amount) * 0.9 + np.random.normal(0, 1.2, n))
delivery_days = np.clip(delivery_days, 1, None).round(0)

product_category = np.random.choice(
    ["Electronics", "Clothing", "Home", "Books", "Sports"], n,
    p=[0.3, 0.25, 0.2, 0.15, 0.1]
)
payment_method = np.random.choice(
    ["Credit Card", "Debit Card", "UPI", "Cash on Delivery"], n,
    p=[0.35, 0.25, 0.3, 0.1]
)

# satisfaction_score: negatively skewed (most customers satisfied, few very low)
satisfaction_score = 100 - np.random.exponential(scale=12, size=n)
satisfaction_score = np.clip(satisfaction_score, 0, 100).round(1)
sat_missing = np.random.choice(n, int(n * 0.05), replace=False)  # small null % (<20%)
satisfaction_score = satisfaction_score.astype(object)
for i in sat_missing:
    satisfaction_score[i] = np.nan

is_returned = np.random.choice([0, 1], n, p=[0.85, 0.15])

df_raw = pd.DataFrame({
    "order_id": order_id,
    "customer_age": customer_age,
    "purchase_amount": purchase_amount,
    "quantity": quantity,
    "discount_percent": discount_percent,
    "customer_rating": customer_rating,
    "delivery_days": delivery_days,
    "product_category": product_category,
    "payment_method": payment_method,
    "satisfaction_score": satisfaction_score,
    "is_returned": is_returned,
})

# inject duplicate rows
dupes = df_raw.sample(15, random_state=1)
df_raw = pd.concat([df_raw, dupes], ignore_index=True)

df_raw.to_csv("ecommerce_raw.csv", index=False)
print("Raw dataset created: ecommerce_raw.csv | shape:", df_raw.shape)


Raw dataset created: ecommerce_raw.csv | shape: (815, 11)


In [3]:
# TASK 1: Load the dataset, inspect shape/dtypes/head

df = pd.read_csv("ecommerce_raw.csv")
print("\n--- TASK 1: Load & Inspect ---")
print(df.head())
print("\nDtypes:\n", df.dtypes)
print("\nShape:", df.shape)




--- TASK 1: Load & Inspect ---
   order_id  customer_age  purchase_amount  quantity discount_percent  \
0         1          40.0            58.15         4               5%   
1         2          34.0            24.30         4               5%   
2         3          41.0            35.08         1              10%   
3         4          50.0            25.09         5              25%   
4         5          33.0            25.52         7               0%   

   customer_rating  delivery_days product_category    payment_method  \
0              5.0           11.0         Clothing        Debit Card   
1              NaN            7.0         Clothing               UPI   
2              4.8            7.0         Clothing       Credit Card   
3              NaN            4.0         Clothing        Debit Card   
4              3.1            5.0      Electronics  Cash on Delivery   

   satisfaction_score  is_returned  
0                99.4            1  
1                99.9 

In [4]:
# TASK 2: Null value analysis

print("\n--- TASK 2: Null Value Analysis ---")
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / df.shape[0]) * 100
null_table = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct.round(2)})
print(null_table)

high_null_cols = null_table[null_table["null_pct"] > 20]
print("\nColumns exceeding 20% null rate:\n", high_null_cols)

low_null_numeric_cols = [
    c for c in df.columns
    if null_table.loc[c, "null_pct"] < 20
    and null_table.loc[c, "null_pct"] > 0
    and pd.api.types.is_numeric_dtype(df[c])
]
for col in low_null_numeric_cols:
    df[col] = df[col].fillna(df[col].median())
print("\nFilled (median) low-null numeric columns:", low_null_numeric_cols)
# Note: customer_rating has >20% nulls -> intentionally left for now,
# handled explicitly later (Task 9) rather than blindly median-filled here.



--- TASK 2: Null Value Analysis ---
                    null_count  null_pct
order_id                     0      0.00
customer_age                 0      0.00
purchase_amount              0      0.00
quantity                     0      0.00
discount_percent            12      1.47
customer_rating            222     27.24
delivery_days                0      0.00
product_category             0      0.00
payment_method               0      0.00
satisfaction_score          40      4.91
is_returned                  0      0.00

Columns exceeding 20% null rate:
                  null_count  null_pct
customer_rating         222     27.24

Filled (median) low-null numeric columns: ['satisfaction_score']


In [5]:
# TASK 3: Duplicate detection and removal

print("\n--- TASK 3: Duplicates ---")
dup_count = df.duplicated().sum()
print("Duplicate rows found:", dup_count)
null_pct_before = df.isnull().sum() / df.shape[0] * 100
df = df.drop_duplicates()
null_pct_after = df.isnull().sum() / df.shape[0] * 100
print("Rows removed:", dup_count, "| New shape:", df.shape)
print("\nNull % change after dedup:\n",
      pd.DataFrame({"before": null_pct_before.round(2), "after": null_pct_after.round(2)}))



--- TASK 3: Duplicates ---
Duplicate rows found: 15
Rows removed: 15 | New shape: (800, 11)

Null % change after dedup:
                     before  after
order_id              0.00    0.0
customer_age          0.00    0.0
purchase_amount       0.00    0.0
quantity              0.00    0.0
discount_percent      1.47    1.5
customer_rating      27.24   27.0
delivery_days         0.00    0.0
product_category      0.00    0.0
payment_method        0.00    0.0
satisfaction_score    0.00    0.0
is_returned           0.00    0.0


In [6]:
# TASK 4: Data type correction

print("\n--- TASK 4: Data Type Correction ---")
print("discount_percent dtype before:", df["discount_percent"].dtype)
df["discount_percent"] = (
    df["discount_percent"].astype(str).str.replace("%", "", regex=False)
)
df["discount_percent"] = pd.to_numeric(df["discount_percent"], errors="coerce")
print("discount_percent dtype after:", df["discount_percent"].dtype)
print("NaNs introduced by coercion (from 'N/A' entries):", df["discount_percent"].isnull().sum())
df["discount_percent"] = df["discount_percent"].fillna(df["discount_percent"].median())

mem_before = df.memory_usage(deep=True).sum()
df["product_category"] = df["product_category"].astype("category")
df["payment_method"] = df["payment_method"].astype("category")
mem_after = df.memory_usage(deep=True).sum()
print(f"\nMemory usage before category conversion: {mem_before} bytes")
print(f"Memory usage after category conversion:  {mem_after} bytes")
print(f"Memory saved: {mem_before - mem_after} bytes")

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")
df["satisfaction_score"] = pd.to_numeric(df["satisfaction_score"], errors="coerce")
df["satisfaction_score"] = df["satisfaction_score"].fillna(df["satisfaction_score"].median())



--- TASK 4: Data Type Correction ---
discount_percent dtype before: object
discount_percent dtype after: float64
NaNs introduced by coercion (from 'N/A' entries): 12

Memory usage before category conversion: 155380 bytes
Memory usage after category conversion:  66459 bytes
Memory saved: 88921 bytes


In [7]:
# TASK 5: Descriptive statistics and skewness

print("\n--- TASK 5: Descriptive Stats & Skewness ---")
print(df.describe())
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if "order_id" in numeric_cols:
    numeric_cols.remove("order_id")
skew_vals = df[numeric_cols].skew().sort_values(key=lambda x: x.abs(), ascending=False)
print("\nSkewness (sorted by |skew|):\n", skew_vals)
top_skew_col = skew_vals.index[0]
print(f"\nColumn with highest absolute skewness: {top_skew_col} (skew={skew_vals.iloc[0]:.3f})")



--- TASK 5: Descriptive Stats & Skewness ---
       order_id  customer_age  purchase_amount   quantity  discount_percent  \
count  800.0000    800.000000       800.000000  800.00000        800.000000   
mean   400.5000     35.068750        43.919888    4.10125         12.118750   
std    231.0844      9.573196        36.016623    2.43329          8.602887   
min      1.0000     18.000000         5.740000    1.00000          0.000000   
25%    200.7500     28.000000        23.375000    3.00000          5.000000   
50%    400.5000     35.000000        34.830000    4.00000         10.000000   
75%    600.2500     41.000000        52.082500    5.00000         20.000000   
max    800.0000     74.000000       383.400000   25.00000         25.000000   

       customer_rating  delivery_days  satisfaction_score  is_returned  
count       584.000000     800.000000          800.000000    800.00000  
mean          3.951199       7.651250           87.630625      0.15250  
std           0.759705 

In [8]:
# TASK 6: Outlier detection with IQR

print("\n--- TASK 6: Outlier Detection (IQR) ---")
outlier_report = {}
for col in ["purchase_amount", "quantity"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {"Q1": Q1, "Q3": Q3, "IQR": IQR, "lower": lower, "upper": upper, "n_outliers": n_out}
    print(f"{col}: Q1={Q1:.2f} Q3={Q3:.2f} IQR={IQR:.2f} bounds=({lower:.2f}, {upper:.2f}) outliers={n_out}")
# Outliers are documented, NOT dropped here (decision explained in README).




--- TASK 6: Outlier Detection (IQR) ---
purchase_amount: Q1=23.38 Q3=52.08 IQR=28.71 bounds=(-19.69, 95.14) outliers=47
quantity: Q1=3.00 Q3=5.00 IQR=2.00 bounds=(0.00, 8.00) outliers=16


In [9]:
# TASK 7: Visualizations

print("\n--- TASK 7: Visualizations (saved to ./plots) ---")

plt.figure(figsize=(10, 4))
plt.plot(df.index, df["purchase_amount"])
plt.title("Purchase Amount by Row Index")
plt.xlabel("Row Index")
plt.ylabel("Purchase Amount")
plt.tight_layout()
plt.savefig("plots/line_purchase_amount.png")
plt.close()

plt.figure(figsize=(8, 5))
df.groupby("product_category")["purchase_amount"].mean().plot(kind="bar")
plt.title("Mean Purchase Amount by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Mean Purchase Amount")
plt.tight_layout()
plt.savefig("plots/bar_mean_purchase_by_category.png")
plt.close()

plt.figure(figsize=(8, 5))
sns.histplot(df[top_skew_col], bins=20)
plt.title(f"Distribution of {top_skew_col} (most skewed column)")
plt.xlabel(top_skew_col)
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("plots/histogram_top_skew.png")
plt.close()

plt.figure(figsize=(8, 5))
sns.scatterplot(x="purchase_amount", y="delivery_days", data=df)
plt.title("Purchase Amount vs Delivery Days")
plt.xlabel("Purchase Amount")
plt.ylabel("Delivery Days")
plt.tight_layout()
plt.savefig("plots/scatter_purchase_vs_delivery.png")
plt.close()

plt.figure(figsize=(8, 5))
sns.boxplot(x="product_category", y="purchase_amount", data=df)
plt.title("Purchase Amount by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Purchase Amount")
plt.tight_layout()
plt.savefig("plots/box_purchase_by_category.png")
plt.close()

print("Saved: line, bar, histogram, scatter, box plots.")



--- TASK 7: Visualizations (saved to ./plots) ---


/tmp/ipykernel_998/589965757.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("product_category")["purchase_amount"].mean().plot(kind="bar")


Saved: line, bar, histogram, scatter, box plots.


In [10]:
# TASK 8: Correlation heat map

print("\n--- TASK 8: Correlation Heat Map ---")
corr_matrix = df[numeric_cols].corr()
print(corr_matrix.round(2))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heat Map (Numeric Columns)")
plt.tight_layout()
plt.savefig("plots/correlation_heatmap.png")
plt.close()

corr_pairs = corr_matrix.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]
top_pair = corr_pairs.index[0]
print(f"\nHighest absolute correlation pair: {top_pair} = {corr_matrix.loc[top_pair[0], top_pair[1]]:.3f}")



--- TASK 8: Correlation Heat Map ---
                    customer_age  purchase_amount  quantity  discount_percent  \
customer_age                1.00             0.01     -0.05              0.01   
purchase_amount             0.01             1.00      0.02              0.01   
quantity                   -0.05             0.02      1.00              0.04   
discount_percent            0.01             0.01      0.04              1.00   
customer_rating            -0.01            -0.12     -0.02              0.06   
delivery_days               0.01             0.82      0.01              0.05   
satisfaction_score          0.06             0.02     -0.05              0.04   
is_returned                 0.01            -0.00      0.03              0.03   

                    customer_rating  delivery_days  satisfaction_score  \
customer_age                  -0.01           0.01                0.06   
purchase_amount               -0.12           0.82                0.02   
quantity  

In [11]:
# TASK 9a: Imputation strategy comparison (two highest-skew columns)

print("\n--- TASK 9a: Imputation Strategy Comparison ---")
top2_skew_cols = skew_vals.index[:2].tolist()
for col in top2_skew_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    print(f"{col}: mean={mean_val:.3f} | median={median_val:.3f} | skew={skew_vals[col]:.3f}")
    if skew_vals[col] > 0:
        df[col] = df[col].fillna(median_val)
        print(f"  -> positively skewed: used MEDIAN to impute (mean is pulled up by high-value outliers)")
    else:
        df[col] = df[col].fillna(median_val)
        print(f"  -> negatively skewed: used MEDIAN to impute (mean is pulled down by low-value outliers)")

print("\nRemaining nulls in these columns:", df[top2_skew_cols].isnull().sum().to_dict())
print("Total remaining nulls in dataset:", df.isnull().sum().sum())

# TASK 9b: Spearman rank correlation vs Pearson

print("\n--- TASK 9b: Spearman vs Pearson ---")
spearman_matrix = df[numeric_cols].corr(method="spearman")
pearson_matrix = corr_matrix  # from Task 8
diff_matrix = (spearman_matrix - pearson_matrix)

print("Spearman matrix:\n", spearman_matrix.round(2))
print("\nPearson matrix (Task 8):\n", pearson_matrix.round(2))

diff_pairs = diff_matrix.abs().unstack().sort_values(ascending=False)
diff_pairs = diff_pairs[diff_pairs.index.get_level_values(0) != diff_pairs.index.get_level_values(1)]
seen = set()
unique_diff_pairs = []
for (a, b), val in diff_pairs.items():
    key = frozenset([a, b])
    if key not in seen:
        seen.add(key)
        unique_diff_pairs.append((a, b, val))

print("\nTop 3 pairs by |Spearman - Pearson|:")
for a, b, val in unique_diff_pairs[:3]:
    sp = spearman_matrix.loc[a, b]
    pe = pearson_matrix.loc[a, b]
    print(f"  {a} vs {b}: Spearman={sp:.3f}, Pearson={pe:.3f}, diff={val:.3f}")


# TASK 9c: Grouped aggregation

print("\n--- TASK 9c: Grouped Aggregation ---")
grouped = df.groupby("product_category")["purchase_amount"].agg(["mean", "std", "count"])
print(grouped)
highest_mean_group = grouped["mean"].idxmax()
highest_std_group = grouped["std"].idxmax()
ratio = grouped["mean"].max() / grouped["mean"].min()
print(f"\nGroup with highest mean: {highest_mean_group}")
print(f"Group with highest std (variance): {highest_std_group}")
print(f"Ratio of highest to lowest group mean: {ratio:.3f}")



--- TASK 9a: Imputation Strategy Comparison ---
purchase_amount: mean=43.920 | median=34.830 | skew=4.284
  -> positively skewed: used MEDIAN to impute (mean is pulled up by high-value outliers)
quantity: mean=4.101 | median=4.000 | skew=3.865
  -> positively skewed: used MEDIAN to impute (mean is pulled up by high-value outliers)

Remaining nulls in these columns: {'purchase_amount': 0, 'quantity': 0}
Total remaining nulls in dataset: 216

--- TASK 9b: Spearman vs Pearson ---
Spearman matrix:
                     customer_age  purchase_amount  quantity  discount_percent  \
customer_age                1.00             0.03     -0.05              0.02   
purchase_amount             0.03             1.00      0.03              0.00   
quantity                   -0.05             0.03      1.00              0.00   
discount_percent            0.02             0.00      0.00              1.00   
customer_rating            -0.02            -0.11      0.01              0.06   
delivery_days

/tmp/ipykernel_998/3132085824.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby("product_category")["purchase_amount"].agg(["mean", "std", "count"])


In [12]:
# TASK 10: Save cleaned dataset

df.to_csv("cleaned_data.csv", index=False)
print("\n--- TASK 10: Saved cleaned_data.csv ---")
print("Final shape:", df.shape)
print("Remaining nulls:", df.isnull().sum().sum())


--- TASK 10: Saved cleaned_data.csv ---
Final shape: (800, 11)
Remaining nulls: 216
